In [1]:
from pathlib import Path
import os
import pandas as pd
from dataclasses import dataclass
from datetime import datetime
import numpy as np
from pydantic import BaseModel
import sys

sys.path.append("..")
import scipy.signal as si
from copy import deepcopy
import json
from classes.sample.raw_data import INCFile, LMSFile

In [2]:
path = Path(
    os.getenv("RAW_DATA_PATH", "data/raw")
)
folders = os.listdir(path)

In [88]:
class Record(BaseModel):
    sampled_at: datetime
    cable: str
    axis: str
    tension: float
    rms: float
    rms_per_band: dict[str, float]

In [89]:
from scipy.signal import butter, filtfilt

In [90]:
def get_rms_per_band(
    signal: list[float], fs: float, f_max: float, bandwidth: float, order: int
) -> dict:
    def get_rms(signal: list[float]) -> float:
        return float(np.sqrt(np.mean(np.square(signal))))

    filters = {}
    for i in range(0, f_max // bandwidth):
        f_start = bandwidth * i
        f_end =  bandwidth * (i + 1)

        if f_start == 0:
            filt = butter(N=order, Wn=f_end, fs=fs, btype="lowpass")
        elif f_end == fs/2:
            filt = butter(N=order, Wn=f_start, fs=fs, btype="highpass")
        else:
            filt = butter(
                N=order,
                Wn=[f_start, f_end],
                fs=fs,
                btype="band",
            )

        filters[f"{bandwidth*i}_{bandwidth*(i+1)}"] = filt

    return {k: get_rms(filtfilt(*filt, signal)) for k, filt in filters.items()}

In [ ]:
def get_features_from_lms(lms: LMSFile, order: int) -> list[Record]:
    def get_rms(x: np.ndarray) -> float:
        return np.sqrt(np.mean(np.square(x)))
    
    sampled_at = lms.sampled_at
    dt = lms.dt
    fs = 1 / dt
    num_samples = len(lms.c2_y)
    records = []
    for cable in ["c1", "c2", "c3", "c4"]:
        tension = np.mean(getattr(lms, f"s{cable[-1]}"))
        for ax in ["x", "y", "z"]:
            signal = np.array(getattr(lms, f"{cable}_{ax}"))
            rms = get_rms(signal)
            
            # filters
            rms_per_band = get_rms_per_band(signal=signal, fs=fs, f_max=128, bandwidth=8, order=5)

            record = Record(
                sampled_at=sampled_at,
                cable=cable,
                axis=ax,
                tension=tension,
                rms=rms,
                rms_per_band=rms_per_band
            )
            records.append(record.dict())
    return records

In [98]:
records = []
count = 0
processed_docs = []
for folder in folders:
    if not folder.startswith("Teste"):
        continue
    files = os.listdir(path / folder)
    prefixes = [f[:-8] for f in files if f.endswith(".txt")]
    prefixes = np.unique(prefixes)
    for prefix in prefixes:
        if prefix in processed_docs:
            continue
        processed_docs.append(prefix)
        lms_path = path / folder / f"{prefix}_LMS.txt"
        try:
            lms_file = LMSFile.from_txt(lms_path)
        except:
            continue
        rec = get_features_from_lms(lms_file, order=20)
        records += rec
        count += 1

In [99]:
records_df = pd.DataFrame(records)
output_path = Path("../data/features")
records_df.to_csv(output_path / "features_band_seq1.csv")